In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s4e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s4e9/train.csv
/kaggle/input/competitions/playground-series-s4e9/test.csv


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor

In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s4e9/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s4e9/test.csv')

In [4]:
print(train_df.head())

   id          brand              model  model_year  milage      fuel_type  \
0   0           MINI      Cooper S Base        2007  213000       Gasoline   
1   1        Lincoln              LS V8        2002  143250       Gasoline   
2   2      Chevrolet  Silverado 2500 LT        2002  136731  E85 Flex Fuel   
3   3        Genesis   G90 5.0 Ultimate        2017   19500       Gasoline   
4   4  Mercedes-Benz        Metris Base        2021    7388       Gasoline   

                                              engine  \
0       172.0HP 1.6L 4 Cylinder Engine Gasoline Fuel   
1       252.0HP 3.9L 8 Cylinder Engine Gasoline Fuel   
2  320.0HP 5.3L 8 Cylinder Engine Flex Fuel Capab...   
3       420.0HP 5.0L 8 Cylinder Engine Gasoline Fuel   
4       208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel   

                     transmission ext_col int_col  \
0                             A/T  Yellow    Gray   
1                             A/T  Silver   Beige   
2                             A/T  

In [5]:
print(train_df.shape)
print(test_df.shape)

(188533, 13)
(125690, 12)


In [6]:
X = train_df.drop(columns=["price"])
y = train_df["price"]

X_test = test_df.copy()

In [7]:
combined = pd.concat([X, X_test], axis=0)

In [8]:
combined.drop(columns=["id"], inplace=True)

In [9]:
for col in combined.columns:

    if combined[col].dtype == "object":
        combined[col] = combined[col].fillna("Missing")

    else:
        combined[col] = combined[col].fillna(combined[col].median())

In [10]:
combined["car_age"] = 2026 - combined["model_year"]

In [11]:
combined["milage_per_year"] = combined["milage"] / combined["car_age"]

In [12]:
combined["engine_size"] = combined["engine"].str.extract(r'(\d+\.\d+)')

In [13]:
combined["engine_size"] = combined["engine_size"].astype(float)

In [14]:
combined["engine_size"] = combined["engine_size"].fillna(
    combined["engine_size"].median()
)

In [15]:
combined["accident"] = combined["accident"].map({
    "None reported": 0,
    "At least 1 accident or damage reported": 1
})

In [16]:
combined["clean_title"] = combined["clean_title"].map({
    "Yes": 1,
    "No": 0
})

In [17]:
cat_features = combined.select_dtypes(include=["object"]).columns.tolist()

In [18]:
print(cat_features)

['brand', 'model', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col']


In [19]:
for col in cat_features:
    combined[col] = combined[col].astype(str)

In [20]:
X = combined.iloc[:len(train_df)]
X_test = combined.iloc[len(train_df):]

In [21]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [22]:
model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=10,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=200
)

In [23]:
model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	learn: 79541.4768656	test: 74262.3375395	best: 74262.3375395 (0)	total: 358ms	remaining: 17m 53s
200:	learn: 71915.2087228	test: 68052.2043414	best: 68050.1366514 (196)	total: 50s	remaining: 11m 36s
400:	learn: 70874.7741422	test: 67978.1015585	best: 67977.4277883 (388)	total: 1m 37s	remaining: 10m 31s
600:	learn: 69908.2149262	test: 67938.9532295	best: 67938.7066387 (599)	total: 2m 29s	remaining: 9m 55s
800:	learn: 69124.7314049	test: 67943.1399703	best: 67938.4807314 (601)	total: 3m 24s	remaining: 9m 22s
1000:	learn: 68239.4249854	test: 67978.8344479	best: 67938.4807314 (601)	total: 4m 21s	remaining: 8m 43s
1200:	learn: 67505.4065843	test: 67993.7954137	best: 67938.4807314 (601)	total: 5m 19s	remaining: 7m 58s
1400:	learn: 66807.6299620	test: 68005.8780007	best: 67938.4807314 (601)	total: 6m 15s	remaining: 7m 8s
1600:	learn: 66242.4825510	test: 68005.3956889	best: 67938.4807314 (601)	total: 7m 10s	remaining: 6m 16s
1800:	learn: 65530.0614849	test: 68029.8295797	best: 67938.4807314

CatBoostRegressor(depth=10, eval_metric='RMSE', iterations=3000, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=200)

In [24]:
valid_preds = model.predict(X_valid)

In [25]:
rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))

print("RMSE:", rmse)

RMSE: 67938.4807314073


In [26]:
test_preds = model.predict(X_test)

In [27]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "price": test_preds
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")

submission.csv created successfully!
